In [ ]:
import numpy as np

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

REGION_TO_ELECTRODES = {
    'Middle_Finger': ['E1', 'E2', 'E3'],
    'Hand':          ['E4', 'E5', 'E6', 'E7'],
    'Forearm':       ['E8', 'E9', 'E10', 'E11', 'E12', 'E13'],
    'Arm':           ['E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20'],
}
ELECTRODE_TO_REGION = {e: r for r, es in REGION_TO_ELECTRODES.items() for e in es}
REGION_TO_LABEL     = {'Middle_Finger': 0, 'Hand': 1, 'Forearm': 2, 'Arm': 3}
LABEL_TO_REGION     = {v: k for k, v in REGION_TO_LABEL.items()}


In [ ]:
from pathlib import Path

BIDS_ROOT   = Path("../../data/BIDS-somatosensory/BIDS-somatosensory")
DERIVATIVES = BIDS_ROOT / "derivatives" / "fmriprep"
subjects    = ["sub-p0002"]
session, task, space = "ses-01", "task-S1Map", "MNI152NLin2009cAsym"
n_runs, HRF_DELAY, WINDOW = 4, 6.0, 1

RESULTS_DIR = Path("results")
FIGURES_DIR = RESULTS_DIR / "figures"
MODELS_DIR  = RESULTS_DIR / "models"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import json

bold_json = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_bold.json"
with open(bold_json, 'r') as f:
    tr = float(json.load(f)["RepetitionTime"])
print(f"TR: {tr} s")


In [ ]:
import pandas as pd

all_events = []
for run in range(1, n_runs + 1):
    path = BIDS_ROOT / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-{run}_events.tsv"
    df = pd.read_csv(path, sep='\t')
    df['run'] = run
    all_events.append(df)

events_df   = pd.concat(all_events, ignore_index=True)
stim_events = events_df[~events_df['trial_type'].isin(['Baseline', 'Jitter'])].copy()
stim_events['region'] = stim_events['trial_type'].map(ELECTRODE_TO_REGION)
print(f"Stimulation events: {len(stim_events)}")


In [ ]:
from nilearn.image import load_img, index_img, new_img_like, resample_to_img
from nilearn.datasets import fetch_atlas_destrieux_2009
from nilearn.maskers import NiftiMasker

first_run_path = DERIVATIVES / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-1_space-{space}_desc-preproc_bold.nii"
first_run_img  = load_img(str(first_run_path))
ref_img        = index_img(first_run_img, 0)

atlas      = fetch_atlas_destrieux_2009()
s1_indices = [i for i, lab in enumerate(atlas.labels) if 'L G_postcentral' in str(lab) and i != 0]
atlas_img  = load_img(atlas.maps)
mask_data  = np.isin(atlas_img.get_fdata(), s1_indices).astype('uint8')
s1_mask    = new_img_like(atlas_img, mask_data)
s1_mask_resampled = resample_to_img(s1_mask, ref_img, interpolation='nearest')
masker = NiftiMasker(mask_img=s1_mask_resampled, standardize=False)
masker.fit(first_run_img)
print(f"Voxels in S1 mask: {int(np.sum(s1_mask_resampled.get_fdata() > 0))}")


In [ ]:
X_list, y_list, run_labels = [], [], []

for run in range(1, n_runs + 1):
    bold_path = DERIVATIVES / subjects[0] / session / "func" / f"{subjects[0]}_{session}_{task}_run-{run}_space-{space}_desc-preproc_bold.nii"
    bold_img  = load_img(str(bold_path))
    run_evs   = stim_events[stim_events['run'] == run]
    for _, event in run_evs.iterrows():
        vol_idx = int(np.round((event["onset"] + HRF_DELAY) / tr))
        vols    = [vol_idx - WINDOW, vol_idx, vol_idx + WINDOW]
        if all(0 <= v < bold_img.shape[3] for v in vols):
            data = [masker.transform(index_img(bold_img, v)) for v in vols]
            data = [d[0] if len(d.shape) == 2 else d for d in data]
            X_list.append(np.mean(data, axis=0))
            y_list.append(event["region"])
            run_labels.append(run)

X          = np.array(X_list)
y_encoded  = np.array([REGION_TO_LABEL[r] for r in y_list])
run_labels = np.array(run_labels)
print(f"X: {X.shape}, y: {y_encoded.shape}")


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

for region, label in REGION_TO_LABEL.items():
    print(f"  {region} (Class {label}): {np.sum(y_encoded == label)} samples")

class_weights  = compute_class_weight('balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weights  = np.sqrt(class_weights)
class_weights /= class_weights.mean()
print("Class weights:", {LABEL_TO_REGION[i]: f"{w:.3f}" for i, w in enumerate(class_weights)})


In [ ]:
mask_vol    = s1_mask_resampled.get_fdata()
xs, ys, zs  = np.where(mask_vol > 0)

x_min, x_max = int(xs.min()), int(xs.max()) + 1
y_min, y_max = int(ys.min()), int(ys.max()) + 1
z_min, z_max = int(zs.min()), int(zs.max()) + 1
bx, by, bz   = x_max - x_min, y_max - y_min, z_max - z_min

def to_3d(X_flat):
    """Reshape (n, n_voxels) flat array → (n, 2, bx, by, bz) 3D bounding box.
    Channel 0: signal; Channel 1: binary mask (constant 1 where real brain is).
    """
    out = np.zeros((len(X_flat), 2, bx, by, bz), dtype=np.float32)
    out[:, 0, xs - x_min, ys - y_min, zs - z_min] = X_flat   # signal
    out[:, 1, xs - x_min, ys - y_min, zs - z_min] = 1.0       # mask
    return out


print(f"S1 bounding box: ({bx}, {by}, {bz}) voxels")
print(f"3D input shape:  (n, 2, {bx}, {by}, {bz})")

In [ ]:
CHANNELS_1   = 32
CHANNELS_2   = 64
DROPOUT      = 0.3
LR           = 0.001
WEIGHT_DECAY = 1e-4
PATIENCE     = 200
BATCH_SIZE   = 16


In [ ]:
import torch.nn.functional as F
from torch.nn import Module, Conv3d, BatchNorm3d, MaxPool3d, AdaptiveAvgPool3d, Linear, Dropout

class SomatotopicCNN(Module):
    def __init__(self, ch1=CHANNELS_1, ch2=CHANNELS_2, dropout=DROPOUT, num_classes=4):
        super().__init__()
        # 2 input channels: signal + binary mask
        self.conv1 = Conv3d(2, ch1, kernel_size=3, padding=1)
        self.bn1   = BatchNorm3d(ch1)
        self.pool1 = MaxPool3d(2)
        # stride-2 conv instead of MaxPool: learns what to downsample
        self.conv2 = Conv3d(ch1, ch2, kernel_size=3, padding=1, stride=2)
        self.bn2   = BatchNorm3d(ch2)
        self.gap    = AdaptiveAvgPool3d(1)
        self.drop   = Dropout(dropout)
        self.fc     = Linear(ch2, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.gap(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)

n_params = sum(p.numel() for p in SomatotopicCNN().parameters())
print(f"Parameters: {n_params:,}")


In [ ]:
import copy
import torch
import torch.optim as optim
from torch.nn import CrossEntropyLoss
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

class_weights_tensor = torch.FloatTensor(class_weights)
criterion            = CrossEntropyLoss(weight=class_weights_tensor)
fold_results         = []

for test_run in range(1, n_runs + 1):
    print(f"\nFold {test_run}/4")
    train_mask = run_labels != test_run
    test_mask  = run_labels == test_run

    scaler      = StandardScaler()
    X_train_s   = scaler.fit_transform(X[train_mask])
    X_test_s    = scaler.transform(X[test_mask])
    y_train     = y_encoded[train_mask]
    y_test      = y_encoded[test_mask]

    X_train_3d  = torch.FloatTensor(to_3d(X_train_s))
    X_test_3d   = torch.FloatTensor(to_3d(X_test_s))
    y_train_t   = torch.LongTensor(y_train)

    train_loader = DataLoader(
        TensorDataset(X_train_3d, y_train_t),
        batch_size=BATCH_SIZE, shuffle=True
    )

    torch.manual_seed(RANDOM_SEED)
    model     = SomatotopicCNN(CHANNELS_1, CHANNELS_2, DROPOUT)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=50, min_lr=1e-6
    )
    best_acc, no_improve, best_state = 0.0, 0, None

    for epoch in range(1000):
        model.train()
        for X_batch, y_batch in train_loader:
            out  = model(X_batch)
            loss = criterion(out, y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            preds = model(X_test_3d).argmax(dim=1).numpy()
        acc = balanced_accuracy_score(y_test, preds)
        scheduler.step(acc)

        if epoch % 50 == 0:
            print(f"  epoch {epoch:3d} | best={best_acc*100:.1f}%", end="\r")

        if acc > best_acc:
            best_acc, no_improve, best_state = acc, 0, copy.deepcopy(model.state_dict())
        else:
            no_improve += 1
        if no_improve >= PATIENCE:
            print(f"  early stop at epoch {epoch} | best={best_acc*100:.1f}%")
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        final_preds = model(X_test_3d).argmax(dim=1).numpy()

    fold_results.append({
        'fold':             test_run,
        'accuracy':         accuracy_score(y_test, final_preds),
        'balanced_accuracy': balanced_accuracy_score(y_test, final_preds),
        'cm':               confusion_matrix(y_test, final_preds),
        'model_state':      best_state,
        'scaler':           scaler,
    })
    print(f"  Acc: {fold_results[-1]['accuracy']*100:.2f}%  |  Bal. Acc: {fold_results[-1]['balanced_accuracy']*100:.2f}%")

mean_acc  = np.mean([r['accuracy']           for r in fold_results])
std_acc   = np.std( [r['accuracy']           for r in fold_results])
mean_bacc = np.mean([r['balanced_accuracy']  for r in fold_results])
std_bacc  = np.std( [r['balanced_accuracy']  for r in fold_results])
print(f"\nMean LORO accuracy         : {mean_acc*100:.2f}% ± {std_acc*100:.2f}%")
print(f"Mean LORO balanced accuracy: {mean_bacc*100:.2f}% ± {std_bacc*100:.2f}%")


In [ ]:
cm_total = np.sum([r['cm'] for r in fold_results], axis=0).astype(int)
np.save(RESULTS_DIR / "cnn_confusion_matrix.npy", cm_total)

best_fold = max(fold_results, key=lambda r: r['balanced_accuracy'])
torch.save(best_fold['model_state'], MODELS_DIR / "best_cnn_model.pt")

pd.DataFrame([{
    'fold':         r['fold'],
    'accuracy':     r['accuracy'],
    'balanced_accuracy': r['balanced_accuracy'],
    'channels_1':   CHANNELS_1,
    'channels_2':   CHANNELS_2,
    'dropout':      DROPOUT,
    'lr':           LR,
    'weight_decay': WEIGHT_DECAY,
} for r in fold_results]).to_csv(RESULTS_DIR / "cnn_results.csv", index=False)

print(f"Best fold: {best_fold['fold']} ({best_fold['balanced_accuracy']*100:.2f}%)")
print(f"Saved → {MODELS_DIR / 'best_cnn_model.pt'}")
print(f"Saved → {RESULTS_DIR / 'cnn_results.csv'}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

labels = [LABEL_TO_REGION[i] for i in range(4)]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_total, annot=True, fmt='d', xticklabels=labels, yticklabels=labels,
            cmap='Blues', ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"CNN — {mean_bacc*100:.1f}% ± {std_bacc*100:.1f}%")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cnn_confusion_matrix.png", dpi=150)
plt.show()
